In [1]:
import os, sys
os.chdir('/home/liangzida/workspace/iTransformer') # 更改工作目录到项目根目录
sys.path.append('/home/liangzida/workspace/iTransformer') # 添加模块路径到 sys.path
import torch
from data_provider.data_loader import Dataset_ETT_hour, Dataset_ETT_minute, Dataset_Custom, Dataset_Solar, Dataset_PEMS, \
    Dataset_Pred
from torch.utils.data import DataLoader

data_dict = {
    'ETTh1': Dataset_ETT_hour,
    'ETTh2': Dataset_ETT_hour,
    'ETTm1': Dataset_ETT_minute,
    'ETTm2': Dataset_ETT_minute,
    'Solar': Dataset_Solar,
    'PEMS': Dataset_PEMS,
    'custom': Dataset_Custom,
}


def data_provider(args, flag):
    Data = data_dict[args.data]
    timeenc = 0 if args.embed != 'timeF' else 1

    if flag == 'test':
        shuffle_flag = False
        drop_last = True
        batch_size = 1  # bsz=1 for evaluation
        freq = args.freq
    elif flag == 'pred':
        shuffle_flag = False
        drop_last = False
        batch_size = 1
        freq = args.freq
        Data = Dataset_Pred
    else:
        shuffle_flag = True
        drop_last = True
        batch_size = args.batch_size  # bsz for train and valid
        freq = args.freq

    class sub_Data(Data):
        def __init__(self, **kwargs):
            super(sub_Data, self).__init__(**kwargs)
            self.data_x = torch.tensor(self.data_x)
            self.data_y = torch.tensor(self.data_y)
            # 判断是否存在self.data_stamp
            if hasattr(self, 'data_stamp'):
                self.data_stamp = torch.tensor(self.data_stamp)

        def __getitem__(self, index):
            global args
            channels = self.data_x.shape[1]
            s_begin = index // channels
            seq_x, seq_y, seq_x_mark, seq_y_mark = super(sub_Data, self).__getitem__(s_begin)
            channel_idx = index % channels
            seq_x = seq_x[:, channel_idx]
            seq_y = seq_y[-args.pred_len:, channel_idx]
            seq_y_mark = seq_y_mark[-args.pred_len:]
            # return torch.tensor(seq_x), torch.tensor(seq_y), torch.tensor(seq_x_mark).expand(-1, 4), torch.tensor(seq_y_mark).expand(-1, 4)
            return seq_x, seq_y, seq_x_mark.expand(-1, 4), seq_y_mark.expand(-1, 4)

        def __len__(self):
            return super(sub_Data, self).__len__() * self.data_x.shape[1]
    Data = sub_Data
    data_set = Data(
        root_path=args.root_path,
        data_path=args.data_path,
        flag=flag,
        size=[args.seq_len, args.label_len, args.pred_len],
        features=args.features,
        target=args.target,
        timeenc=timeenc,
        freq=freq,
    )
    print(flag, len(data_set))
    data_loader = DataLoader(
        data_set,
        batch_size=batch_size,
        shuffle=shuffle_flag,
        num_workers=args.num_workers,
        drop_last=drop_last)
    return data_set, data_loader


/home/liangzida/anaconda3/envs/cnn/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# from data_provider.data_factory import data_provider

class Args():
    def __init__(self):
        self.embed = 'timeF'
        self.target = 'OT'
        self.label_len = 1
        self.freq = 'h'

        self.data = 'custom'
        self.root_path = './dataset/electricity/'
        self.data_path = 'electricity.csv'
        self.features = 'M'
        self.enc_in = 321
        self.dec_in = 321

        self.seq_len = 24
        self.pred_len = 24
        self.batch_size = 16
        self.num_workers = 0

dataset_configs = [
    {'root_path':'./dataset/traffic/', 'data_path':'traffic.csv', 'features':'M', 'enc_in':862, 'dec_in':862},
    {'root_path':'./dataset/weather/', 'data_path':'weather.csv', 'features':'M', 'enc_in':21, 'dec_in':21},
    {'root_path':'./dataset/ETT-small/', 'data_path':'ETTh1.csv', 'data':'ETTh1', 'features':'M', 'enc_in':7, 'dec_in':7},
    {'root_path':'./dataset/ETT-small/', 'data_path':'ETTh2.csv', 'data':'ETTh2', 'features':'M', 'enc_in':7, 'dec_in':7},
    {'root_path':'./dataset/ETT-small/', 'data_path':'ETTm1.csv', 'data':'ETTm1', 'features':'M', 'enc_in':7, 'dec_in':7},
    {'root_path':'./dataset/ETT-small/', 'data_path':'ETTm2.csv', 'data':'ETTm2', 'features':'M', 'enc_in':7, 'dec_in':7},
    {'root_path':'./dataset/PEMS/', 'data_path':'PEMS03.npz', 'data':'PEMS', 'features':'M', 'enc_in':358, 'dec_in':358},
    {'root_path':'./dataset/PEMS/', 'data_path':'PEMS04.npz', 'data':'PEMS', 'features':'M', 'enc_in':307, 'dec_in':307},
    {'root_path':'./dataset/PEMS/', 'data_path':'PEMS07.npz', 'data':'PEMS', 'features':'M', 'enc_in':883, 'dec_in':883},
    {'root_path':'./dataset/PEMS/', 'data_path':'PEMS08.npz', 'data':'PEMS', 'features':'M', 'enc_in':170, 'dec_in':170},
    {'root_path':'./dataset/Solar/', 'data_path':'solar_AL.txt', 'data':'Solar', 'features':'M', 'enc_in':137, 'dec_in':137},
]

datasets = []
for config in dataset_configs:
    args = Args()
    args.__dict__.update(config)
    data_set, data_loader = data_provider(args, 'train')
    datasets.append(data_set)

train 10544846
train 773640
train 60151
train 60151
train 241591
train 241591
train 5612366
train 3115436
train 14911221
train 1813220
train 5034065


In [3]:
type(datasets[0][0][0]), datasets[0][0][0].shape, datasets[0][0][1].shape, datasets[0][0][2].shape, datasets[0][0][3].shape

(torch.Tensor,
 torch.Size([24]),
 torch.Size([24]),
 torch.Size([24, 4]),
 torch.Size([24, 4]))

In [4]:
from torch.utils.data import ConcatDataset
from tqdm import tqdm

# 将datasets合并成一个大的dataset
combined_dataset = ConcatDataset(datasets)

index = 0
for i in [10544846, 773640, 60151, 60151, 241591, 241591, 5612366, 3115436, 14911221, 1813220, 5034065]:
    print(index, combined_dataset[index][0].shape, combined_dataset[index][1].shape, combined_dataset[index][2].shape, combined_dataset[index][3].shape)
    index += i


0 torch.Size([24]) torch.Size([24]) torch.Size([24, 4]) torch.Size([24, 4])
10544846 torch.Size([24]) torch.Size([24]) torch.Size([24, 4]) torch.Size([24, 4])
11318486 torch.Size([24]) torch.Size([24]) torch.Size([24, 4]) torch.Size([24, 4])
11378637 torch.Size([24]) torch.Size([24]) torch.Size([24, 4]) torch.Size([24, 4])
11438788 torch.Size([24]) torch.Size([24]) torch.Size([24, 4]) torch.Size([24, 4])
11680379 torch.Size([24]) torch.Size([24]) torch.Size([24, 4]) torch.Size([24, 4])
11921970 torch.Size([24]) torch.Size([24]) torch.Size([24, 4]) torch.Size([24, 4])
17534336 torch.Size([24]) torch.Size([24]) torch.Size([24, 4]) torch.Size([24, 4])
20649772 torch.Size([24]) torch.Size([24]) torch.Size([24, 4]) torch.Size([24, 4])
35560993 torch.Size([24]) torch.Size([24]) torch.Size([24, 4]) torch.Size([24, 4])
37374213 torch.Size([24]) torch.Size([24]) torch.Size([24, 4]) torch.Size([24, 4])


In [5]:
combined_loader = torch.utils.data.DataLoader(combined_dataset, batch_size=16, shuffle=True, num_workers=10, drop_last=True)
for i, data in tqdm(enumerate(combined_loader), total=len(combined_loader)):
    pass

  1%|          | 13943/2650517 [00:25<1:20:57, 542.77it/s]


KeyboardInterrupt: 

In [ ]:
from torch.utils.data import DataLoader

class CustomDataLoader(DataLoader):
    def __init__(self, datasets, batch_size=1, shuffle=False, num_workers=0, drop_last=False):
        combined_dataset = ConcatDataset(datasets)
        super(CustomDataLoader, self).__init__(combined_dataset, batch_size=batch_size, shuffle=shuffle, num_workers=num_workers, drop_last=drop_last)

# Example usage
custom_loader = CustomDataLoader(datasets, batch_size=16, shuffle=True, num_workers=10, drop_last=True)

for i, data in enumerate(custom_loader):
    print(i, data)
    if i == 10:
        break